# CT Single Run In Colab

This notebook runs one configurable CT reconstruction experiment in Colab.

By default it uses the small repo-safe TIFF CT subset in `demo-samples/ct_l067_subset_tiff`, so it can run without uploading local `.IMA` files.

## Runtime

In Colab, go to `Runtime > Change runtime type` and choose:
- `Python 3`
- `GPU`

Prefer `A100` when available.

In [ ]:
#@title Project Settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"  #@param {type:"string"}
REPO_BRANCH = "codex-pdhg-colab-light-100"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"  #@param {type:"string"}

REPO_DIR = "/content/mycode2"  #@param {type:"string"}
PYTHON_BIN = "/usr/bin/python3"  #@param {type:"string"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/ct_single_run_exports"  #@param {type:"string"}
DRIVE_CT_DATA_DIR = ""  #@param {type:"string"}
DRIVE_CHECKPOINT_DIR = ""  #@param {type:"string"}
HF_MODEL_ID = "jiayangshi/lodochallenge_pixel_diffusion"  #@param {type:"string"}

RUN_NAME = "CT_Single_Run"  #@param {type:"string"}
CONFIG_NAME = "default_ct.yaml"  #@param ["default_ct.yaml", "default_ct_admm.yaml"]
SESSION_TAG = ""  #@param {type:"string"}

SEED = 99  #@param {type:"integer"}
TOTAL_IMAGES = 1  #@param {type:"integer"}
BATCH_SIZE = 1  #@param {type:"integer"}
DATA_START_IDX = 1  #@param {type:"integer"}
DATA_END_IDX = 2  #@param {type:"integer"}

NUM_STEPS = 400  #@param {type:"integer"}
MAX_ITER = 400  #@param {type:"integer"}
SIGMA_MAX = 10.0  #@param {type:"number"}
SIGMA_MIN = 0.075  #@param {type:"number"}
TAU = 0.01  #@param {type:"number"}
SIGMA_DUAL = 1200.0  #@param {type:"number"}
I0 = 10000.0  #@param {type:"number"}
NUM_ANGLES = 80  #@param {type:"integer"}
ADMM_LGVD_NUM_STEPS = 10  #@param {type:"integer"}

EVAL_METRICS = "psnr;ssim"  #@param {type:"string"}
SAVE_SAMPLES = True  #@param {type:"boolean"}
SAVE_TRAJ = False  #@param {type:"boolean"}
SAVE_TRAJ_RAW_DATA = False  #@param {type:"boolean"}
SHOW_CONFIG = False  #@param {type:"boolean"}
LOG_TAIL_LINES = 120  #@param {type:"integer"}

# Optional extra Hydra overrides, separated by semicolons.
EXTRA_HYDRA_OVERRIDES = ""  #@param {type:"string"}


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Fetch The Repo
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
repo_dir.parent.mkdir(parents=True, exist_ok=True)
os.chdir(repo_dir.parent)

if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR
    ], check=True)
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as handle:
        handle.extractall(repo_dir.parent)
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

os.chdir(REPO_DIR)
subprocess.run(["git", "status", "--short"], check=True)


In [ ]:
#@title Install Dependencies
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-r", "requirements-colab-ct.txt"], check=True)


In [ ]:
#@title Optional: Copy Local Checkpoint From Drive
import os
import shutil
from pathlib import Path

os.chdir(REPO_DIR)
local_checkpoint_path = None
if DRIVE_CHECKPOINT_DIR.strip():
    src = Path(DRIVE_CHECKPOINT_DIR)
    if not src.exists():
        raise FileNotFoundError(f"Checkpoint folder not found: {src}")
    dst = Path(REPO_DIR) / "pretrained-models" / "dm4ct" / "lodochallenge_pixel_diffusion"
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    local_checkpoint_path = dst.as_posix()
    print(f"Copied checkpoint to: {dst}")
else:
    print("Using Hugging Face model download.")


In [ ]:
#@title Build Run Command
import json
import os
import shlex
from datetime import datetime
from pathlib import Path

os.chdir(REPO_DIR)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
session_bits = [RUN_NAME.strip(), SESSION_TAG.strip(), timestamp]
run_tag = "_".join(bit for bit in session_bits if bit)
run_tag = run_tag.replace(" ", "_")
save_root = Path(REPO_DIR) / "results" / "colab_ct_single" / run_tag
hydra_root = save_root / "hydra"
run_aux_root = save_root / "run_aux"
latest_log_path = run_aux_root / "run.log"
latest_pid_path = run_aux_root / "run.pid"

overrides = [
    f"seed={SEED}",
    f"name={RUN_NAME}",
    f"total_images={TOTAL_IMAGES}",
    f"batch_size={BATCH_SIZE}",
    f"save_dir={save_root.as_posix()}",
    f"save_samples={str(SAVE_SAMPLES).lower()}",
    f"save_traj={str(SAVE_TRAJ).lower()}",
    f"save_traj_raw_data={str(SAVE_TRAJ_RAW_DATA).lower()}",
    f"show_config={str(SHOW_CONFIG).lower()}",
    f"eval_fn_list=[{','.join(EVAL_METRICS.split(';'))}]",
    f"sampler.annealing_scheduler_config.num_steps={NUM_STEPS}",
    f"sampler.annealing_scheduler_config.sigma_max={SIGMA_MAX}",
    f"sampler.annealing_scheduler_config.sigma_min={SIGMA_MIN}",
    f"inverse_task.admm_config.max_iter={MAX_ITER}",
    f"inverse_task.admm_config.pdhg.tau={TAU}",
    f"inverse_task.admm_config.pdhg.sigma_dual={SIGMA_DUAL}",
    f"inverse_task.operator.I0={I0}",
    f"inverse_task.operator.num_angles={NUM_ANGLES}",
    f"model.model_config.local_files_only={'true' if local_checkpoint_path else 'false'}",
]

if local_checkpoint_path:
    overrides.append(f"model.model_config.model_id={local_checkpoint_path}")
else:
    overrides.append(f"model.model_config.model_id={HF_MODEL_ID}")

if CONFIG_NAME == "default_ct_admm.yaml":
    overrides.append(f"inverse_task.admm_config.denoise.lgvd.num_steps={ADMM_LGVD_NUM_STEPS}")

if DRIVE_CT_DATA_DIR.strip():
    drive_data_dir = Path(DRIVE_CT_DATA_DIR)
    if not drive_data_dir.exists():
        raise FileNotFoundError(f"CT data folder not found: {drive_data_dir}")
    overrides.extend([
        f"data.image_root_path={drive_data_dir.as_posix()}",
        f"data.start_idx={DATA_START_IDX}",
        f"data.end_idx={DATA_END_IDX}",
    ])
else:
    repo_tiff_subset = Path(REPO_DIR) / "demo-samples" / "ct_l067_subset_tiff"
    overrides.extend([
        f"data.image_root_path={repo_tiff_subset.as_posix()}",
        f"data.start_idx={DATA_START_IDX}",
        f"data.end_idx={DATA_END_IDX}",
    ])

if EXTRA_HYDRA_OVERRIDES.strip():
    overrides.extend([item.strip() for item in EXTRA_HYDRA_OVERRIDES.split(';') if item.strip()])

run_cmd = [
    PYTHON_BIN,
    "recover_inverse2.py",
    "--config-name",
    CONFIG_NAME,
    f"hydra.run.dir={hydra_root.as_posix()}",
] + overrides

print("Command:\n")
print(" ".join(shlex.quote(part) for part in run_cmd))

last_context = {
    "run_tag": run_tag,
    "save_root": save_root.as_posix(),
    "hydra_root": hydra_root.as_posix(),
    "latest_log_path": latest_log_path.as_posix(),
    "latest_pid_path": latest_pid_path.as_posix(),
    "run_cmd": run_cmd,
}

run_aux_root.mkdir(parents=True, exist_ok=True)
context_path = run_aux_root / "context.json"
context_path.write_text(json.dumps(last_context, indent=2), encoding="utf-8")
print(f"Context saved to: {context_path}")


In [ ]:
#@title Launch The Run In Background
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
latest_log_path = Path(last_context["latest_log_path"])
latest_pid_path = Path(last_context["latest_pid_path"])
latest_log_path.parent.mkdir(parents=True, exist_ok=True)

with latest_log_path.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(
        last_context["run_cmd"],
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

latest_pid_path.write_text(str(process.pid), encoding="utf-8")
print(f"PID: {process.pid}")
print(f"Log: {latest_log_path}")
print(f"Save root: {last_context['save_root']}")


In [ ]:
#@title Show Recent Log Lines
from pathlib import Path

log_path = Path(last_context["latest_log_path"])
if not log_path.exists():
    raise FileNotFoundError(f"Log file not found: {log_path}")

lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
tail = lines[-int(LOG_TAIL_LINES):]
print("\n".join(tail) if tail else "<log is empty>")


In [ ]:
#@title Show Results And Artifacts
import json
from pathlib import Path

save_root = Path(last_context["save_root"])
metrics_matches = sorted(save_root.rglob("metrics.json"))
metric_history_matches = sorted(save_root.rglob("metric_history.json"))
eval_matches = sorted(save_root.rglob("eval.md"))
grid_matches = sorted(save_root.rglob("grid_results.png"))

print("Artifacts:\n")
print(f"save_root: {save_root}")
print(f"metrics.json matches: {[p.as_posix() for p in metrics_matches]}")
print(f"metric_history.json matches: {[p.as_posix() for p in metric_history_matches]}")
print(f"eval.md matches: {[p.as_posix() for p in eval_matches]}")
print(f"grid_results.png matches: {[p.as_posix() for p in grid_matches]}")

if eval_matches:
        eval_path = eval_matches[0]
        print("\neval.md:\n")
        print(eval_path.read_text(encoding="utf-8", errors="ignore"))
else:
        print("\nNo eval.md found yet.")

if metrics_matches:
        metrics_path = metrics_matches[0]
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        print("\nmetrics.json:\n")
        print(json.dumps(metrics, indent=2))
else:
        print("\nNo metrics.json found yet.")

if metric_history_matches:
        metric_history_path = metric_history_matches[0]
        metric_history = json.loads(metric_history_path.read_text(encoding="utf-8"))
        print("\nmetric_history.json summary:\n")
        print(f"path: {metric_history_path}")
        if isinstance(metric_history, dict) and "runs" in metric_history:
            print(f"runs captured: {len(metric_history['runs'])}")
            history_view = metric_history["runs"][0] if metric_history["runs"] else {}
        else:
            history_view = metric_history
        series_keys = [key for key, value in history_view.items() if isinstance(value, list)]
        print(f"series keys: {series_keys}")
        for key in series_keys:
            series = history_view[key]
            if not series:
                print(f"{key}: <empty>")
                continue
            tail = series[-5:] if len(series) > 5 else series
            print(f"{key}: len={len(series)} final={series[-1]} tail={tail}")
else:
        print("\nNo metric_history.json found yet.")


In [ ]:
#@title Copy Run Artifacts To Drive
import shutil
from pathlib import Path

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)

targets = [
    Path(last_context["save_root"]),
    Path(last_context["hydra_root"]),
    Path(last_context["latest_log_path"]),
]

for src in targets:
    if not src.exists():
        print(f"Skipping missing path: {src}")
        continue

    dst = export_root / src.name
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
